# HTN 规划与进化搜索

符号规划管「计划必须可证明正确」；进化代码搜索管「适应度可机器校验」。ChatHTN 与 AlphaEvolve 各自把 LLM 当成放大器，而不是替代规划器或评估器。

## 问题描述

ReWOO、Plan-and-Execute、ReAct 覆盖了多数 agent 规划。两类它们处理不好：

1. **需要可证明正确的计划。** 调度、合规流程——LLM 偶尔幻觉一步就不可接受。
2. **有机器可校验适应度的优化。** 矩阵乘法、调度启发式、编译器 pass——目标不是「正确计划」，而是「更好方案」。

HTN / ChatHTN 与 AlphaEvolve 分别对症这两类问题。

## 基本概念

### Hierarchical Task Networks（HTN）

HTN 的核心部件：
- **任务** ———— 复合任务（需分解）与原子任务（可直接执行）。
- **方法** ———— 把复合任务拆成子任务，带前置条件。
- **算子** ———— 原子动作，带前置条件与效果。
- **状态** ———— 一组事实。

规划：给定目标任务与初始状态，找到一条分解路径，使原子算子按序满足前置条件。HTN 早于 LLM，至今仍是「可证明正确计划」的参照。

### ChatHTN：符号搜索 + LLM 回退

ChatHTN 在符号 HTN 与 LLM 之间交错：

1. 先用已有方法分解当前复合任务。
2. 没有适用方法时，问 LLM：「在状态 `s` 下如何分解 `task`？」
3. 把 LLM 回答译成候选子任务。
4. 对照算子 schema 校验；非法分解直接拒绝。
5. 递归继续。

**核心主张：** 产出的每条计划在符号层可证明健全——LLM 只提供候选分解，从不直接改写计划。正确性归符号层；LLM 扩展方法库。

后续工作（online method learning）会把 LLM 分解泛化进方法库，显著降低重复查询。

### AlphaEvolve：进化循环 + 程序化评估

AlphaEvolve 是另一条线：用 LLM 集成做变异，用确定性评估器做选择。

循环：
1. 种子程序 + 程序化评估器（返回适应度分数）。
2. LLM 集成提出变异。
3. 评估器跑变异。
4. 保留更优个体，再变异。

**硬约束：** 适应度必须可机器校验、确定性、足够快。「让 LLM 判断代码是否更好」不是适应度函数——对散文答案做进化搜索不会收敛。

### 何时用哪个

| 问题类型 | 用 | 为什么 |
|----------|----|--------|
| 硬约束调度 / 合规 | HTN + ChatHTN | 可证明健全 |
| 带测试 / 基准的代码优化 | AlphaEvolve | 测试即评估器 |
| 多步任务执行 | ReAct / ReWOO | 无形式保证即可 |
| 策略绑定自动化 | HTN | 前置条件编码策略 |

多数 agent 任务不需要二者——先 ReAct / ReWOO。

### 什么时候这种模式失败

- **没有算子 schema 的 HTN。** 没有前置/效果，健全性主张崩塌；ChatHTN 依赖 schema 拒绝非法分解。
- **没有真评估器的 AlphaEvolve。** 「问 LLM 好不好」不是适应度；评估器必须确定性且快。
- **过度工程。** 多数任务用 ReAct / ReWOO 就够。

# 开始编码

对应本章核心：**HTN 符号规划（任务 / 方法 / 算子 / 状态）**、**ChatHTN（符号优先 + LLM 回退 + schema 校验）**、**AlphaEvolve（变异 + 确定性适应度）**。  
先用玩具域跑通可证明分解与进化循环；再用 **PyTorch** 学方法选择 / 适应度代理；最后用 **LangChain + DeepSeek** 做真实 ChatHTN 回退与进化变异。


## 1. 教学玩具：HTN + ChatHTN + AlphaEvolve

微型物流域：复合任务靠方法分解；原子算子带前置与效果。ChatHTN 在缺方法时用脚本化 LLM 提案，schema 拒绝非法分解。AlphaEvolve 用确定性评估器优化调度启发式。


In [ ]:
from __future__ import annotations

import random
import re
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

TaskKind = Literal["compound", "primitive"]


@dataclass(frozen=True)
class Atom:
    """一阶事实，如 ``at(truck, A)``。"""

    pred: str
    args: tuple[str, ...]

    def __str__(self) -> str:
        return f"{self.pred}({','.join(self.args)})"


def parse_atom(text: str) -> Atom:
    """
    Args:
        text: ``pred(a,b)`` 形式。

    Returns:
        atom: 解析结果。
    """
    m = re.fullmatch(r"\s*([A-Za-z_][\w]*)\(([^)]*)\)\s*", text)
    if not m:
        raise ValueError(f"bad atom: {text}")
    args = tuple(a.strip() for a in m.group(2).split(",") if a.strip())
    return Atom(m.group(1), args)


@dataclass
class State:
    """状态 = 事实集合。"""

    facts: set[Atom] = field(default_factory=set)

    def copy(self) -> State:
        return State(set(self.facts))

    def holds(self, atom: Atom) -> bool:
        return atom in self.facts

    def apply(self, add: list[Atom], delete: list[Atom]) -> None:
        for a in delete:
            self.facts.discard(a)
        for a in add:
            self.facts.add(a)

    def render(self) -> str:
        return "{" + ", ".join(sorted(map(str, self.facts))) + "}"


@dataclass
class Operator:
    """原子算子：前置 + 效果。"""

    name: str
    params: list[str]
    preconds: list[str]
    add: list[str]
    delete: list[str]

    def ground(self, binding: dict[str, str]) -> tuple[list[Atom], list[Atom], list[Atom]]:
        """
        Args:
            binding: 参数绑定。

        Returns:
            pre, add, delete: 接地原子列表。
        """

        def sub(template: str) -> Atom:
            out = template
            for k, v in binding.items():
                out = out.replace(f"?{k}", v)
            return parse_atom(out)

        return (
            [sub(p) for p in self.preconds],
            [sub(a) for a in self.add],
            [sub(d) for d in self.delete],
        )

    def applicable(self, state: State, binding: dict[str, str]) -> bool:
        pre, _, _ = self.ground(binding)
        return all(state.holds(p) for p in pre)

    def apply(self, state: State, binding: dict[str, str]) -> State:
        if not self.applicable(state, binding):
            raise ValueError(f"operator {self.name} not applicable with {binding}")
        _, add, delete = self.ground(binding)
        ns = state.copy()
        ns.apply(add, delete)
        return ns


@dataclass
class Method:
    """复合任务分解方法。"""

    name: str
    task: str
    preconds: list[str]
    subtasks: list[str]


@dataclass
class Task:
    """规划任务节点。"""

    name: str
    args: tuple[str, ...]
    kind: TaskKind

    def signature(self) -> str:
        return f"{self.name}({','.join(self.args)})"


def parse_task(text: str, primitives: set[str]) -> Task:
    """
    Args:
        text: ``name(a,b)``。
        primitives: 原子任务名集合。

    Returns:
        task: 复合或原子。
    """
    atom = parse_atom(text)
    kind: TaskKind = "primitive" if atom.pred in primitives else "compound"
    return Task(atom.pred, atom.args, kind)


@dataclass
class HTNDomain:
    """算子 schema + 方法库。"""

    operators: dict[str, Operator]
    methods: list[Method]

    @property
    def primitive_names(self) -> set[str]:
        return set(self.operators)

    def methods_for(self, task: Task, state: State) -> list[tuple[Method, dict[str, str], list[Task]]]:
        """
        Returns:
            candidates: ``(method, binding, subtasks)``。
        """
        out: list[tuple[Method, dict[str, str], list[Task]]] = []
        for m in self.methods:
            tm = re.fullmatch(r"([A-Za-z_]+)\((.*)\)", m.task.replace(" ", ""))
            if not tm or tm.group(1) != task.name:
                continue
            params = [p.strip() for p in tm.group(2).split(",") if p.strip()]
            if len(params) != len(task.args):
                continue
            binding: dict[str, str] = {}
            ok = True
            for p, a in zip(params, task.args):
                if p.startswith("?"):
                    binding[p[1:]] = a
                elif p != a:
                    ok = False
                    break
            if not ok:
                continue

            def sub_atom(template: str) -> Atom:
                s = template
                for k, v in binding.items():
                    s = s.replace(f"?{k}", v)
                return parse_atom(s)

            if not all(state.holds(sub_atom(p)) for p in m.preconds):
                continue

            def sub_task(template: str) -> Task:
                s = template
                for k, v in binding.items():
                    s = s.replace(f"?{k}", v)
                return parse_task(s, self.primitive_names)

            subs = [sub_task(st) for st in m.subtasks]
            out.append((m, binding, subs))
        return out

    def validate_decomposition(
        self,
        task: Task,
        subtask_sigs: list[str],
        state: State,
    ) -> tuple[bool, str, list[Task]]:
        """
        对照算子/任务 schema 校验 LLM 提案。

        Returns:
            ok, reason, subtasks。
        """
        del state  # 轻量 schema 校验不依赖状态
        if task.kind != "compound":
            return False, "not a compound task", []
        allowed = self.primitive_names | {"deliver", "get_to"}
        subs: list[Task] = []
        try:
            for sig in subtask_sigs:
                t = parse_task(sig, self.primitive_names)
                if t.name not in allowed:
                    return False, f"schema reject: unknown task {t.name}", []
                if t.name not in self.primitive_names:
                    t.kind = "compound"
                subs.append(t)
        except ValueError as e:
            return False, f"schema reject: {e}", []
        if not subs:
            return False, "schema reject: empty decomposition", []
        return True, "ok", subs


def logistics_domain() -> HTNDomain:
    """构建玩具物流域。"""
    ops = {
        "drive": Operator(
            name="drive",
            params=["truck", "from", "to"],
            preconds=["at(?truck,?from)", "road(?from,?to)"],
            add=["at(?truck,?to)"],
            delete=["at(?truck,?from)"],
        ),
        "load": Operator(
            name="load",
            params=["pkg", "truck", "loc"],
            preconds=["at(?pkg,?loc)", "at(?truck,?loc)"],
            add=["in(?pkg,?truck)"],
            delete=["at(?pkg,?loc)"],
        ),
        "unload": Operator(
            name="unload",
            params=["pkg", "truck", "loc"],
            preconds=["in(?pkg,?truck)", "at(?truck,?loc)"],
            add=["at(?pkg,?loc)"],
            delete=["in(?pkg,?truck)"],
        ),
    }
    methods = [
        Method(
            name="deliver_from_A",
            task="deliver(?pkg,?dest)",
            preconds=["at(?pkg,A)", "at(t1,A)", "road(A,?dest)"],
            subtasks=[
                "load(?pkg,t1,A)",
                "drive(t1,A,?dest)",
                "unload(?pkg,t1,?dest)",
            ],
        ),
    ]
    return HTNDomain(ops, methods)


@dataclass
class PlanStep:
    """计划中的一步原子动作。"""

    op: str
    binding: dict[str, str]

    def render(self) -> str:
        return f"{self.op}({','.join(self.binding.values())})"


@dataclass
class ChatHTNPlanner:
    """符号方法优先；缺方法时 LLM 回退；schema 校验；可学习方法库。"""

    domain: HTNDomain
    llm_decompose: Callable[[Task, State], list[str]]
    learned_methods: list[Method] = field(default_factory=list)
    max_depth: int = 16

    def _domain(self) -> HTNDomain:
        return HTNDomain(self.domain.operators, self.domain.methods + self.learned_methods)

    def plan(self, goals: list[Task], state: State) -> tuple[list[PlanStep] | None, list[str]]:
        """
        Returns:
            plan: 原子步骤或 ``None``。
            log: 搜索轨迹。
        """
        log: list[str] = []
        plan: list[PlanStep] = []
        ok = self._seek(list(goals), state.copy(), plan, log, 0)
        return (plan if ok else None), log

    def _seek(
        self,
        tasks: list[Task],
        state: State,
        plan: list[PlanStep],
        log: list[str],
        depth: int,
    ) -> bool:
        if depth > self.max_depth:
            log.append("fail: max depth")
            return False
        if not tasks:
            return True
        task, rest = tasks[0], tasks[1:]
        domain = self._domain()

        if task.kind == "primitive":
            op = domain.operators.get(task.name)
            if op is None:
                log.append(f"fail: no operator {task.name}")
                return False
            binding = dict(zip(op.params, task.args))
            if not op.applicable(state, binding):
                log.append(f"fail: precond {task.signature()} in {state.render()}")
                return False
            plan.append(PlanStep(task.name, binding))
            ns = op.apply(state, binding)
            log.append(f"op {task.signature()}")
            return self._seek(rest, ns, plan, log, depth + 1)

        cands = domain.methods_for(task, state)
        base_len = len(plan)
        for method, _binding, subs in cands:
            del plan[base_len:]
            log.append(f"method {method.name} -> {[s.signature() for s in subs]}")
            if self._seek(subs + rest, state.copy(), plan, log, depth + 1):
                return True
        del plan[base_len:]

        log.append(f"llm_fallback for {task.signature()}")
        proposal = self.llm_decompose(task, state)
        ok, reason, subs = domain.validate_decomposition(task, proposal, state)
        if not ok:
            log.append(reason)
            return False
        grounded = Method(
            name=f"learned_{task.signature()}",
            task=task.signature(),
            preconds=[],
            subtasks=[s.signature() for s in subs],
        )
        # 避免重复学习同一接地方法
        if not any(m.name == grounded.name for m in self.learned_methods):
            self.learned_methods.append(grounded)
            log.append(f"learned method {grounded.name} -> {grounded.subtasks}")
        return self._seek(subs + rest, state, plan, log, depth + 1)


def scripted_llm_decompose(task: Task, state: State) -> list[str]:
    """玩具 LLM：为 ``get_to`` / 跨站 ``deliver`` 提候选；可故意含非法算子供校验拒绝。"""
    if task.name == "get_to" and len(task.args) == 2:
        truck, dest = task.args
        loc = None
        for f in state.facts:
            if f.pred == "at" and f.args[0] == truck:
                loc = f.args[1]
                break
        if loc is None or loc == dest:
            return ["teleport(t1,moon)"]
        return [f"drive({truck},{loc},{dest})"]
    if task.name == "deliver" and len(task.args) == 2:
        pkg, dest = task.args
        pkg_loc = truck_loc = None
        for f in state.facts:
            if f.pred == "at" and f.args[0] == pkg:
                pkg_loc = f.args[1]
            if f.pred == "at" and f.args[0] == "t1":
                truck_loc = f.args[1]
        if pkg_loc and truck_loc and pkg_loc == truck_loc:
            return [
                f"load({pkg},t1,{pkg_loc})",
                f"get_to(t1,{dest})",
                f"unload({pkg},t1,{dest})",
            ]
        if pkg_loc and truck_loc:
            return [
                f"get_to(t1,{pkg_loc})",
                f"load({pkg},t1,{pkg_loc})",
                f"get_to(t1,{dest})",
                f"unload({pkg},t1,{dest})",
            ]
    return ["teleport(pkg,moon)"]


def simulate(plan: list[PlanStep], state: State, domain: HTNDomain) -> State:
    """执行计划验证健全性。"""
    s = state.copy()
    for step in plan:
        op = domain.operators[step.op]
        s = op.apply(s, step.binding)
    return s


@dataclass
class Individual:
    """进化个体：启发式权重。"""

    genes: dict[str, float]
    fitness: float = float("-inf")
    code: str = ""

    def render_code(self) -> str:
        return (
            "def score(job):\n"
            f"    return {self.genes.get('w_urgent', 1.0):.3f}*job['urgent'] "
            f"+ {self.genes.get('w_short', 1.0):.3f}*(1.0/max(job['duration'],1)) "
            f"- {self.genes.get('w_late', 1.0):.3f}*job['late_risk']\n"
        )


def eval_scheduler(genes: dict[str, float], jobs: list[dict[str, float]]) -> float:
    """
    确定性适应度：加权延误的相反数（越高越好）。

    Returns:
        fitness: 机器可校验分数。
    """
    w_u = genes.get("w_urgent", 1.0)
    w_s = genes.get("w_short", 1.0)
    w_l = genes.get("w_late", 1.0)

    def key(job: dict[str, float]) -> float:
        return (
            w_u * job["urgent"]
            + w_s * (1.0 / max(job["duration"], 1.0))
            - w_l * job["late_risk"]
        )

    order = sorted(jobs, key=key, reverse=True)
    t = 0.0
    lateness = 0.0
    for job in order:
        t += job["duration"]
        lateness += max(0.0, t - job["deadline"]) * (1.0 + job["urgent"])
    return -lateness


def mutate_genes(parent: dict[str, float], rng: random.Random) -> dict[str, float]:
    """玩具变异（生产由 LLM 集成提案）。"""
    child = dict(parent)
    key = rng.choice(list(child))
    child[key] = max(0.05, child[key] + rng.uniform(-0.4, 0.4))
    return child


@dataclass
class AlphaEvolve:
    """进化循环：变异 → 程序化评估 → 保留更优。"""

    jobs: list[dict[str, float]]
    population: list[Individual] = field(default_factory=list)
    rng: random.Random = field(default_factory=lambda: random.Random(0))

    def seed(self, n: int = 4) -> None:
        for _ in range(n):
            g = {
                "w_urgent": self.rng.uniform(0.2, 2.0),
                "w_short": self.rng.uniform(0.2, 2.0),
                "w_late": self.rng.uniform(0.2, 2.0),
            }
            ind = Individual(genes=g, fitness=eval_scheduler(g, self.jobs))
            ind.code = ind.render_code()
            self.population.append(ind)
        self.population.sort(key=lambda x: x.fitness, reverse=True)

    def step(self) -> Individual:
        parent = self.population[0]
        child_g = mutate_genes(parent.genes, self.rng)
        child = Individual(genes=child_g, fitness=eval_scheduler(child_g, self.jobs))
        child.code = child.render_code()
        self.population.append(child)
        self.population.sort(key=lambda x: x.fitness, reverse=True)
        self.population = self.population[:5]
        return self.population[0]

    def run(self, generations: int = 20) -> Individual:
        if not self.population:
            self.seed()
        best = self.population[0]
        for _ in range(generations):
            best = self.step()
        return best


print("HTN/ChatHTN/AlphaEvolve toy ready")


## 2. 玩具示例：可证明交付计划、非法分解拒绝、进化调度


In [ ]:
def demo_htn_and_evolve() -> None:
    """ChatHTN 交付包裹；拒绝 teleport；AlphaEvolve 降低延误。"""
    domain = logistics_domain()
    state = State(
        {
            parse_atom("at(t1,A)"),
            parse_atom("at(p1,A)"),
            parse_atom("road(A,B)"),
            parse_atom("road(B,A)"),
        }
    )
    planner = ChatHTNPlanner(domain, scripted_llm_decompose)
    goal = [parse_task("deliver(p1,B)", domain.primitive_names)]
    plan, log = planner.plan(goal, state)
    print("=== ChatHTN plan ===")
    assert plan is not None, log
    for step in plan:
        print(step.render())
    print("log tail:", log[-6:])
    final = simulate(plan, state, domain)
    assert final.holds(parse_atom("at(p1,B)"))
    print("final:", final.render())

    ok, reason, _ = domain.validate_decomposition(
        parse_task("deliver(p1,B)", domain.primitive_names),
        ["teleport(p1,B)"],
        state,
    )
    print("\n=== reject illegal ===", ok, reason)
    assert not ok

    plan2, log2 = planner.plan(goal, state)
    assert plan2 is not None
    assert any(("learned_" in x) or ("method" in x) for x in log2)

    # 跨站：只有 A->B，目标 C 时需 LLM get_to 链（扩展路网）
    state_c = State(
        {
            parse_atom("at(t1,A)"),
            parse_atom("at(p1,B)"),
            parse_atom("road(A,B)"),
            parse_atom("road(B,A)"),
            parse_atom("road(B,C)"),
            parse_atom("road(C,B)"),
        }
    )
    planner2 = ChatHTNPlanner(domain, scripted_llm_decompose)
    plan_c, log_c = planner2.plan(
        [parse_task("deliver(p1,C)", domain.primitive_names)], state_c
    )
    print("\n=== cross-city via llm_fallback ===")
    print("ok", plan_c is not None, "log:", [x for x in log_c if "llm" in x or "learned" in x or "reject" in x][:6])
    if plan_c:
        for step in plan_c:
            print(step.render())
        assert simulate(plan_c, state_c, domain).holds(parse_atom("at(p1,C)"))

    jobs = [
        {"urgent": 0.9, "duration": 5, "deadline": 6, "late_risk": 0.8},
        {"urgent": 0.2, "duration": 2, "deadline": 10, "late_risk": 0.1},
        {"urgent": 0.5, "duration": 3, "deadline": 8, "late_risk": 0.4},
        {"urgent": 0.7, "duration": 4, "deadline": 7, "late_risk": 0.6},
    ]
    evo = AlphaEvolve(jobs=jobs)
    evo.seed(5)
    start_fit = evo.population[0].fitness
    best = evo.run(30)
    print("\n=== AlphaEvolve ===")
    print("start_fit", start_fit, "best_fit", best.fitness)
    print(best.code)
    assert best.fitness >= start_fit
    print("TOY DEMO OK")


demo_htn_and_evolve()


## 3. PyTorch：方法适用性 + 适应度代理

1. 状态特征 → 是否该用某方法 / 走 LLM 回退。  
2. 基因向量 → 预测调度适应度（**真选择仍用确定性评估器**）。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MethodPolicy(nn.Module):
    """状态袋 → 方法 logits（含 llm_fallback）。"""

    def __init__(self, n_feat: int = 6, n_act: int = 3) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_feat, 16), nn.ReLU(), nn.Linear(16, n_act))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        single = x.ndim == 1
        if single:
            x = x.unsqueeze(0)
        y = self.net(x)
        return y.squeeze(0) if single else y


class FitnessSurrogate(nn.Module):
    """基因 → 适应度代理（不替代真评估器）。"""

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        single = x.ndim == 1
        if single:
            x = x.unsqueeze(0)
        y = self.net(x).squeeze(-1)
        return y.squeeze(0) if single else y


METHOD_LABELS = ["deliver_direct", "need_get_to", "llm_fallback"]


def state_features(
    same_city: bool, has_road: bool, pkg_loaded: bool, truck_at_pkg: bool
) -> torch.Tensor:
    """
    Returns:
        x: ``(6,)`` 特征。
    """
    return torch.tensor(
        [
            1.0 if same_city else 0.0,
            1.0 if has_road else 0.0,
            1.0 if pkg_loaded else 0.0,
            1.0 if truck_at_pkg else 0.0,
            1.0 if same_city and has_road else 0.0,
            1.0 if (not truck_at_pkg) else 0.0,
        ],
        dtype=torch.float32,
    )


def train_method_policy(steps: int = 300) -> MethodPolicy:
    data = [
        (state_features(True, True, False, True), 0),
        (state_features(False, True, False, False), 1),
        (state_features(False, False, False, False), 2),
        (state_features(True, True, False, True), 0),
        (state_features(False, True, True, True), 1),
        (state_features(False, False, True, False), 2),
    ]
    model = MethodPolicy()
    opt = torch.optim.Adam(model.parameters(), lr=0.05)
    model.train()
    for _ in range(steps):
        losses = []
        for x, y in data:
            logits = model(x)
            losses.append(F.cross_entropy(logits.unsqueeze(0), torch.tensor([y])))
        loss = torch.stack(losses).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model


def train_fitness_surrogate(
    samples: list[tuple[dict[str, float], float]],
    steps: int = 400,
) -> FitnessSurrogate:
    model = FitnessSurrogate()
    opt = torch.optim.Adam(model.parameters(), lr=0.05)
    xs = [
        torch.tensor([g["w_urgent"], g["w_short"], g["w_late"]], dtype=torch.float32)
        for g, _ in samples
    ]
    Y = torch.tensor([fit for _, fit in samples], dtype=torch.float32)
    X = torch.stack(xs)
    y_mean, y_std = Y.mean(), Y.std().clamp_min(1e-3)
    Yn = (Y - y_mean) / y_std
    model.train()
    for _ in range(steps):
        pred = model(X)
        loss = F.mse_loss(pred, Yn)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model


def demo_pytorch_htn_evolve() -> None:
    torch.manual_seed(0)
    policy = train_method_policy()
    with torch.no_grad():
        logits = policy(state_features(True, True, False, True))
        choice = METHOD_LABELS[int(torch.argmax(logits).item())]
    print("=== method policy ===")
    print("same_city+road ->", choice, logits.tolist())
    assert choice == "deliver_direct"

    jobs = [
        {"urgent": 0.9, "duration": 5, "deadline": 6, "late_risk": 0.8},
        {"urgent": 0.2, "duration": 2, "deadline": 10, "late_risk": 0.1},
        {"urgent": 0.5, "duration": 3, "deadline": 8, "late_risk": 0.4},
    ]
    samples: list[tuple[dict[str, float], float]] = []
    rng = random.Random(1)
    for _ in range(40):
        g = {
            "w_urgent": rng.uniform(0.1, 2.0),
            "w_short": rng.uniform(0.1, 2.0),
            "w_late": rng.uniform(0.1, 2.0),
        }
        samples.append((g, eval_scheduler(g, jobs)))
    sur = train_fitness_surrogate(samples)
    g_good = max(samples, key=lambda t: t[1])[0]
    g_bad = min(samples, key=lambda t: t[1])[0]
    s_good = float(
        sur(torch.tensor([g_good["w_urgent"], g_good["w_short"], g_good["w_late"]]))
    )
    s_bad = float(
        sur(torch.tensor([g_bad["w_urgent"], g_bad["w_short"], g_bad["w_late"]]))
    )
    print("=== fitness surrogate ===")
    print("proxy good/bad", s_good, s_bad)
    assert s_good > s_bad
    print("PYTORCH DEMO OK")


demo_pytorch_htn_evolve()


## 4. 生产级：LangChain ChatHTN + AlphaEvolve（DeepSeek）

工具：``htn_plan``、``propose_decomposition``（schema 门禁）、``evolve_step``（确定性 fitness）。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"

PROD_DOMAIN = logistics_domain()
PROD_STATE = State(
    {
        parse_atom("at(t1,A)"),
        parse_atom("at(p1,A)"),
        parse_atom("road(A,B)"),
        parse_atom("road(B,C)"),
        parse_atom("road(C,B)"),
        parse_atom("road(B,A)"),
    }
)
PROD_PLANNER = ChatHTNPlanner(PROD_DOMAIN, scripted_llm_decompose)
PROD_JOBS = [
    {"urgent": 0.9, "duration": 5, "deadline": 6, "late_risk": 0.8},
    {"urgent": 0.2, "duration": 2, "deadline": 10, "late_risk": 0.1},
    {"urgent": 0.5, "duration": 3, "deadline": 8, "late_risk": 0.4},
    {"urgent": 0.7, "duration": 4, "deadline": 7, "late_risk": 0.6},
]
PROD_EVO = AlphaEvolve(jobs=PROD_JOBS)


class PlanArgs(BaseModel):
    """htn_plan。"""

    goal: str = Field(description="Goal task like deliver(p1,B)")


class ProposeDecompArgs(BaseModel):
    """propose_decomposition。"""

    task: str = Field(description="Compound task signature")
    subtasks: list[str] = Field(description="Candidate subtask signatures in order")


class EvolveArgs(BaseModel):
    """evolve_step。"""

    generations: int = Field(default=15, ge=1, le=50)


def htn_plan_impl(goal: str) -> str:
    task = parse_task(goal, PROD_DOMAIN.primitive_names)
    plan, log = PROD_PLANNER.plan([task], PROD_STATE)
    if plan is None:
        return json.dumps({"ok": False, "log": log[-12:]}, ensure_ascii=False)
    return json.dumps(
        {
            "ok": True,
            "plan": [s.render() for s in plan],
            "log_tail": log[-8:],
            "learned_methods": [m.name for m in PROD_PLANNER.learned_methods],
        },
        ensure_ascii=False,
    )


def propose_decomposition_impl(task: str, subtasks: list[str]) -> str:
    t = parse_task(task, PROD_DOMAIN.primitive_names)
    ok, reason, subs = PROD_DOMAIN.validate_decomposition(t, subtasks, PROD_STATE)
    if not ok:
        return json.dumps({"accepted": False, "reason": reason}, ensure_ascii=False)
    method = Method(
        name=f"learned_{t.signature()}",
        task=t.signature(),
        preconds=[],
        subtasks=[s.signature() for s in subs],
    )
    if not any(m.name == method.name for m in PROD_PLANNER.learned_methods):
        PROD_PLANNER.learned_methods.append(method)
    return json.dumps(
        {"accepted": True, "method": method.name, "subtasks": method.subtasks},
        ensure_ascii=False,
    )


def evolve_step_impl(generations: int = 15) -> str:
    if not PROD_EVO.population:
        PROD_EVO.seed(5)
    start = PROD_EVO.population[0].fitness
    best = PROD_EVO.run(generations)
    return json.dumps(
        {
            "start_fitness": start,
            "best_fitness": best.fitness,
            "genes": best.genes,
            "code": best.code,
            "note": "fitness is deterministic eval_scheduler; LLM must not judge quality",
        },
        ensure_ascii=False,
    )


def build_tools() -> list[StructuredTool]:
    def _plan(**kwargs: Any) -> str:
        return htn_plan_impl(PlanArgs(**kwargs).goal)

    def _propose(**kwargs: Any) -> str:
        a = ProposeDecompArgs(**kwargs)
        return propose_decomposition_impl(a.task, a.subtasks)

    def _evo(**kwargs: Any) -> str:
        return evolve_step_impl(EvolveArgs(**kwargs).generations)

    return [
        StructuredTool.from_function(
            name="htn_plan",
            description="Symbolic HTN plan for a goal; uses learned methods; LLM never edits the plan directly.",
            func=_plan,
            args_schema=PlanArgs,
        ),
        StructuredTool.from_function(
            name="propose_decomposition",
            description=(
                "Submit a candidate decomposition for a compound task. "
                "Validated against operator/task schema; illegal ops like teleport are rejected."
            ),
            func=_propose,
            args_schema=ProposeDecompArgs,
        ),
        StructuredTool.from_function(
            name="evolve_step",
            description="Run AlphaEvolve generations with deterministic fitness on scheduling genes.",
            func=_evo,
            args_schema=EvolveArgs,
        ),
    ]


TOOLS = build_tools()


def reset_prod() -> None:
    global PROD_DOMAIN, PROD_STATE, PROD_PLANNER, PROD_EVO, TOOLS
    PROD_DOMAIN = logistics_domain()
    PROD_STATE = State(
        {
            parse_atom("at(t1,A)"),
            parse_atom("at(p1,A)"),
            parse_atom("road(A,B)"),
            parse_atom("road(B,C)"),
            parse_atom("road(C,B)"),
            parse_atom("road(B,A)"),
        }
    )
    PROD_PLANNER = ChatHTNPlanner(PROD_DOMAIN, scripted_llm_decompose)
    PROD_EVO = AlphaEvolve(jobs=list(PROD_JOBS))
    TOOLS = build_tools()


def get_llm(*, temperature: float = 0.0) -> Any:
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def build_agent() -> Any:
    system = (
        "You help with HTN planning and AlphaEvolve-style optimization.\n"
        "- For delivery goals, call htn_plan. If it fails, propose_decomposition with valid "
        "operators only: drive/load/unload/get_to/deliver — never teleport.\n"
        "- For scheduling optimization, call evolve_step. Do NOT claim a program is better "
        "without the tool fitness.\n"
        "Reply in Chinese, concise."
    )
    return create_agent(get_llm(), TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args')})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            lines.append(f"OBS[{m.name}]: {m.content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


def run_agent(user_text: str) -> dict[str, Any]:
    return build_agent().invoke({"messages": [HumanMessage(content=user_text)]})


print(f"LangChain HTN/Evolve ready | {MODEL}")


## 5. 生产示例：规划交付 + 拒绝非法分解 + 进化一步


In [ ]:
def demo_deepseek_htn_evolve() -> None:
    """真实 API；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    reset_prod()

    r1 = run_agent("请用 htn_plan 规划 deliver(p1,B)，并简述计划步骤。")
    print("=== plan ===")
    print(format_agent_messages(r1["messages"]))
    assert count_tool_calls(r1["messages"]) >= 1

    r2 = run_agent(
        "尝试 propose_decomposition：task=deliver(p1,C)，子任务用 teleport(p1,C)。"
        "若被拒绝，改成合法的 drive/load/unload/get_to 分解再提交。"
    )
    print("\n=== schema gate ===")
    print(format_agent_messages(r2["messages"]))
    blob = format_agent_messages(r2["messages"]).lower()
    assert "accepted" in blob or "reject" in blob or "teleport" in blob

    r3 = run_agent("对调度启发式跑 evolve_step(generations=20)，报告起止 fitness。")
    print("\n=== evolve ===")
    print(format_agent_messages(r3["messages"]))
    assert count_tool_calls(r3["messages"]) >= 1
    print("\nPRODUCTION DEMO OK")


demo_deepseek_htn_evolve()
